# Linear Regression — Statistics

**Goal.** Treat the OLS estimator $\hat{\theta}$ as a *random* vector that depends on the noisy targets y, and quantify its statistical behaviour: bias, variance, optimality (Gauss–Markov), sampling distribution under normal noise, confidence intervals, hypothesis tests, the $R^2$ decomposition, and what happens when assumptions break.

**Role of this notebook.** Estimator theory: math first, with **minimal** Monte Carlo simulations used only to *verify* a theorem numerically. The deterministic linear-algebra picture (normal equations, hat matrix, pseudoinverse) is in `02_mathematics.ipynb`; the optimisation algorithms in `03_optimization.ipynb`; the full implementation in `05_hands_on_programming.ipynb`.

**Prerequisites.** `02_mathematics.ipynb` — in particular the closed form $\theta^* = (X^\top X)^{-1} X^\top y$ (Theorem 4.2), the hat matrix H (Theorem 5.3), and Pythagoras $\|y\|^2 = \|\hat{y}^*\|^2 + \|r^*\|^2$ (Corollary 5.4).

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → **`04_statistics`** → `05_hands_on_programming`.

**Eight questions.**

1. What is random and what is fixed? (The statistical setup.)
2. Under what assumptions is $\hat{\theta}$ a meaningful estimator? (Gauss–Markov A1–A4.)
3. Is $\hat{\theta}$ on average correct? (Unbiasedness.)
4. How much does $\hat{\theta}$ vary across resamples? (Variance formula.)
5. Is $\hat{\theta}$ the *best* linear unbiased estimator? (Gauss–Markov theorem.)
6. What is its full distribution? (Adding Gaussianity, A5.)
7. How do we test hypotheses and build confidence intervals?
8. How much variance does the model explain? ($R^2$ via Pythagoras.)

---

**Reading conventions.** Same as `02_mathematics.ipynb`: theorems in blockquotes, derivations in code blocks, equations numbered only when referred to later. Code cells run small Monte Carlo experiments that *verify* a theorem — never derive one.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. The statistical setup

In `02_mathematics.ipynb` the targets y were just *numbers*. To do statistics we must say where those numbers come from. The standard **fixed-design** setup of linear regression goes:

- The design matrix $X \in \mathbb{R}^{n \times p}$ is **fixed** (not random). We condition on it throughout — all expectations and variances are conditional on X.
- There exists an **unknown true parameter** $\theta \in \mathbb{R}^p$. We never see $\theta$; we only see y.
- The targets $y \in \mathbb{R}^n$ are **random**, generated by

> $$y = X\theta + \varepsilon, \quad \text{where } \varepsilon \in \mathbb{R}^n \text{ is a random noise vector.} \tag{1.1}$$

The OLS estimator

> $$\hat{\theta} := (X^\top X)^{-1} X^\top y \quad \text{(assuming } \operatorname{rank}(X) = p\text{; see Theorem 4.2 of \texttt{02\_mathematics.ipynb})}$$

is a random vector because it is a (deterministic) function of the random vector y. Its randomness is inherited entirely from $\varepsilon$.

**What we want.** A list of *properties* of $\hat{\theta}$ — its mean, variance, distribution — written in terms of the *assumptions* we put on $\varepsilon$. Sections 2–6 do exactly that.

## 2. The Gauss–Markov assumptions

Classical regression theory is organised around five assumptions on $(X, \varepsilon)$. The first four are the **Gauss–Markov** assumptions; the fifth (normality) is needed only for exact distributional results.

| Label | Name | Statement |
|---|---|---|
| **A1** | Linearity in parameters | $y = X\theta + \varepsilon$ with $X \in \mathbb{R}^{n \times p}$ fixed, $\operatorname{rank}(X) = p$. |
| **A2** | Strict exogeneity (zero mean noise) | $\mathbb{E}[\varepsilon] = 0$. |
| **A3** | Homoscedasticity | $\operatorname{Var}(\varepsilon_i) = \sigma^2$ for every $i$. |
| **A4** | No autocorrelation | $\operatorname{Cov}(\varepsilon_i, \varepsilon_j) = 0$ for $i \neq j$. |
| **A5** | Normality (optional) | $\varepsilon \sim \mathcal{N}(0, \sigma^2 I_n)$ |

**Compact restatement of A2 + A3 + A4.** All three together are equivalent to a single matrix identity on the covariance of the noise:

> $$\mathbb{E}[\varepsilon] = 0, \quad \operatorname{Var}(\varepsilon) := \mathbb{E}[\varepsilon \varepsilon^\top] = \sigma^2 I_n. \tag{2.1}$$

**Compact restatement of A2 + A3 + A4 + A5.** Adding normality gives the joint distribution

> $$\varepsilon \sim \mathcal{N}(0, \sigma^2 I_n) \tag{2.2}$$

(2.2) is exactly the assumption of Theorem 2.2 in `02_mathematics.ipynb` that made OLS = MLE.

## 3. Unbiasedness

> **Theorem 3.1 (Unbiasedness).** Under A1 and A2,
>
> $$\mathbb{E}[\hat{\theta}] = \theta.$$

**Proof.** Substitute $y = X\theta + \varepsilon$ from (1.1) into the closed form:

$$\begin{aligned}
\hat{\theta} &= (X^\top X)^{-1} X^\top y \\
  &= (X^\top X)^{-1} X^\top (X\theta + \varepsilon) \\
  &= (X^\top X)^{-1} (X^\top X) \theta + (X^\top X)^{-1} X^\top \varepsilon \\
  &= \theta + (X^\top X)^{-1} X^\top \varepsilon. \tag{3.1}
\end{aligned}$$

Equation (3.1) is the **sampling identity**: $\hat{\theta}$ equals the truth plus a *linear function of the noise*. Take expectation, using linearity and A2 ($\mathbb{E}[\varepsilon] = 0$):

$$\mathbb{E}[\hat{\theta}] = \theta + (X^\top X)^{-1} X^\top \cdot \mathbb{E}[\varepsilon] = \theta + 0 = \theta. \quad \blacksquare$$

**Reading.** OLS does not systematically over- or under-estimate. Average over infinitely many fresh draws of y (with X held fixed) and you recover $\theta$ exactly. This is what *unbiased* means; it says nothing about how close any single $\hat{\theta}$ is to $\theta$ — that is variance, §4.

### 3.1 Monte Carlo verification

Pick a true $\theta$, draw M independent y's, fit OLS each time, and check the empirical mean of $\hat{\theta}$ across draws is close to $\theta$.

In [ ]:
n, p = 200, 3
theta_true = np.array([1.5, -2.0, 0.7])
sigma = 0.5

# Fixed design (drawn once and held throughout).
X = np.column_stack([np.ones(n), rng.normal(size=n), rng.normal(size=n)])

M = 5000  # number of Monte Carlo replications
thetas = np.empty((M, p))
for m in range(M):
    eps = rng.normal(0, sigma, size=n)
    y = X @ theta_true + eps
    thetas[m] = np.linalg.solve(X.T @ X, X.T @ y)

emp_mean = thetas.mean(axis=0)
print(f"true theta            = {theta_true}")
print(f"empirical mean theta_hat = {emp_mean}")
print(f"max abs deviation     = {np.max(np.abs(emp_mean - theta_true)):.4f}")

**Reading.** With M = 5000 draws the deviation is on the order of $10^{-2}$, consistent with the $O(1/\sqrt{M})$ Monte Carlo error. The closer the empirical mean to $\theta_{\text{true}}$, the more confidently we say "$\hat{\theta}$ is unbiased".

## 4. Variance of $\hat{\theta}$

> **Theorem 4.1 (Sandwich identity).** Under A1 + A2 + A3 + A4,
>
> $$\operatorname{Var}(\hat{\theta}) = \sigma^2 (X^\top X)^{-1} \tag{4.1}$$

**Proof.** From the sampling identity (3.1), $\hat{\theta} - \theta = A\varepsilon$ where $A := (X^\top X)^{-1} X^\top$. Standard rule for the variance of a linear function of a random vector:

$$\operatorname{Var}(\hat{\theta}) = \operatorname{Var}(A\varepsilon) = A \cdot \operatorname{Var}(\varepsilon) \cdot A^\top.$$

By (2.1), $\operatorname{Var}(\varepsilon) = \sigma^2 I_n$. Therefore

$$\begin{aligned}
\operatorname{Var}(\hat{\theta}) &= A \cdot (\sigma^2 I_n) \cdot A^\top \\
  &= \sigma^2 \cdot (X^\top X)^{-1} X^\top \cdot X (X^\top X)^{-1} \\
  &= \sigma^2 \cdot (X^\top X)^{-1} \cdot (X^\top X) \cdot (X^\top X)^{-1} \\
  &= \sigma^2 \cdot (X^\top X)^{-1}. \quad \blacksquare
\end{aligned}$$

**Reading.** Three knobs control the variance of the $j$-th coefficient — $\operatorname{Var}(\hat{\theta}_j) = \sigma^2 \cdot [(X^\top X)^{-1}]_{jj}$:

- **$\sigma^2$.** More noise in y ⇒ more noise in $\hat{\theta}$. Linear.
- **Sample size n.** Hidden in $X^\top X$. If columns of X have bounded entries, $(X^\top X) \propto n$, so $(X^\top X)^{-1} \propto 1/n$ and the standard error of each coefficient shrinks like $1/\sqrt{n}$.
- **Multicollinearity.** When two columns of X are nearly proportional, $X^\top X$ is ill-conditioned and $(X^\top X)^{-1}$ has huge diagonal entries — the **variance inflation** phenomenon.

### 4.1 Monte Carlo verification

Compute the empirical covariance of $\hat{\theta}$ across the M draws from §3.1 and compare to the theoretical $\sigma^2 (X^\top X)^{-1}$.

In [ ]:
emp_cov = np.cov(thetas, rowvar=False)
theo_cov = sigma**2 * np.linalg.inv(X.T @ X)

print("empirical Var(theta_hat):")
print(emp_cov)
print("\ntheoretical sigma^2 (X^T X)^-1:")
print(theo_cov)
print(f"\nmax abs entrywise difference = {np.max(np.abs(emp_cov - theo_cov)):.5f}")

## 5. The Gauss–Markov theorem: OLS is BLUE

Unbiasedness alone is not impressive — the trivial estimator "always predict 0" is also a (very bad) function of y. The Gauss–Markov theorem makes a sharper claim: among **all** linear unbiased estimators, OLS has the smallest variance, coordinate by coordinate.

### 5.1 Definitions

> **Definition (linear estimator).** An estimator $\tilde{\theta}$ of $\theta$ is **linear (in y)** if there exists a matrix $C \in \mathbb{R}^{p \times n}$ (depending on X but not on y) such that
>
> $$\tilde{\theta} = Cy.$$

OLS is linear with $C_{\text{OLS}} := (X^\top X)^{-1} X^\top$.

> **Definition (unbiased).** $\tilde{\theta} = Cy$ is **unbiased** if $\mathbb{E}[\tilde{\theta}] = \theta$ for every value of $\theta \in \mathbb{R}^p$. Substituting $y = X\theta + \varepsilon$ and $\mathbb{E}[\varepsilon] = 0$, this is equivalent to the algebraic condition
>
> $$CX = I_p \tag{5.1}$$

**Comparison rule.** For two $p \times p$ covariance matrices $\Sigma_1, \Sigma_2$, we write $\Sigma_1 \preceq \Sigma_2$ if $\Sigma_2 - \Sigma_1$ is PSD. Setting $v = e_j$ in $v^\top (\Sigma_2 - \Sigma_1) v \ge 0$ shows that $\Sigma_1 \preceq \Sigma_2$ implies the $j$-th diagonal of $\Sigma_1$ is no larger than that of $\Sigma_2$ — every individual coordinate has smaller (or equal) variance.

### 5.2 Theorem (Gauss–Markov)

> **Theorem 5.2.** Under A1 + A2 + A3 + A4, the OLS estimator $\hat{\theta}$ is **BLUE** — the *best linear unbiased estimator*. Concretely: for any linear unbiased $\tilde{\theta} = Cy$,
>
> $$\operatorname{Var}(\hat{\theta}) \preceq \operatorname{Var}(\tilde{\theta})$$
>
> with equality iff $C = C_{\text{OLS}} = (X^\top X)^{-1} X^\top$.

**Proof.** Write $C = C_{\text{OLS}} + D$ where $D := C - C_{\text{OLS}}$. Unbiasedness of both $\hat{\theta}$ and $\tilde{\theta}$ requires $CX = I_p$ and $C_{\text{OLS}} X = I_p$, hence

$$DX = CX - C_{\text{OLS}} X = I_p - I_p = 0. \tag{5.2}$$

Compute $\operatorname{Var}(\tilde{\theta})$ by the same rule as in §4 ($\operatorname{Var}(C\varepsilon) = \sigma^2 CC^\top$ since $\operatorname{Var}(\varepsilon) = \sigma^2 I_n$):

$$\begin{aligned}
\operatorname{Var}(\tilde{\theta}) &= \sigma^2 \cdot CC^\top \\
  &= \sigma^2 \cdot (C_{\text{OLS}} + D)(C_{\text{OLS}} + D)^\top \\
  &= \sigma^2 \cdot \bigl[ C_{\text{OLS}} C_{\text{OLS}}^\top + C_{\text{OLS}} D^\top + D C_{\text{OLS}}^\top + DD^\top \bigr].
\end{aligned}$$

The cross terms vanish because (5.2) gives $DX = 0$:

$$\begin{aligned}
C_{\text{OLS}} D^\top &= (X^\top X)^{-1} X^\top D^\top = (X^\top X)^{-1} (DX)^\top = 0, \\
D C_{\text{OLS}}^\top &= (C_{\text{OLS}} D^\top)^\top = 0.
\end{aligned}$$

Also $C_{\text{OLS}} C_{\text{OLS}}^\top = (X^\top X)^{-1}$ (computed already in §4). Hence

$$\operatorname{Var}(\tilde{\theta}) = \sigma^2 \cdot (X^\top X)^{-1} + \sigma^2 \cdot DD^\top = \operatorname{Var}(\hat{\theta}) + \sigma^2 \cdot DD^\top.$$

$DD^\top$ is PSD (it is a Gram matrix), so $\operatorname{Var}(\tilde{\theta}) - \operatorname{Var}(\hat{\theta}) \succeq 0$ — exactly the claim. Equality forces $DD^\top = 0$, hence $D = 0$, hence $C = C_{\text{OLS}}$. $\blacksquare$

**Reading.** Gauss–Markov is sharp: OLS is *the* unique BLUE estimator. The catch is the word **linear** — biased or non-linear estimators (e.g., Ridge regression, the Lasso) can have **smaller** variance, at the price of bias. That trade-off is the subject of the bias–variance decomposition (`00_foundations/04_model_evaluation/`) and motivates the next algorithms in module 01.

## 6. Estimating $\sigma^2$ and the sampling distribution

The variance formula (4.1) contains $\sigma^2$, which we do not know. We estimate it from the residuals.

### 6.1 Definition (residual sum of squares and residual variance)

Let $\hat{r} := y - X\hat{\theta} \in \mathbb{R}^n$ be the fitted residual vector. Define

> $$\text{RSS} := \|\hat{r}\|^2 = \sum_i \hat{r}_i^2,$$
>
> $$s^2 := \text{RSS} / (n - p). \tag{6.1}$$

### 6.2 Theorem (unbiased estimator of $\sigma^2$)

> **Theorem 6.2.** Under A1 + A2 + A3 + A4, with $\operatorname{rank}(X) = p$,
>
> $$\mathbb{E}[s^2] = \sigma^2, \quad \text{so } s^2 \text{ is an unbiased estimator of } \sigma^2.$$

**Proof sketch.** Using $\hat{r} = (I_n - H)y$ (with H the hat matrix from §5 of `02_mathematics.ipynb`) and the trace identity $\mathbb{E}[\varepsilon^\top M \varepsilon] = \sigma^2 \cdot \operatorname{trace}(M)$ for symmetric M and $\varepsilon$ with covariance $\sigma^2 I_n$:

$$\mathbb{E}[\text{RSS}] = \mathbb{E}[\varepsilon^\top (I_n - H) \varepsilon] = \sigma^2 \cdot \operatorname{trace}(I_n - H) = \sigma^2 \cdot (n - p),$$

since $\operatorname{trace}(H) = \operatorname{rank}(H) = p$ by Theorem 5.3 of `02_mathematics.ipynb`. Dividing by $(n - p)$ gives $\mathbb{E}[s^2] = \sigma^2$. $\blacksquare$

**Reading.** The divisor is **$n - p$**, not $n$. Each fitted parameter "absorbs" one residual degree of freedom: the $n$ residuals satisfy $p$ linear constraints ($X^\top \hat{r} = 0$ from the normal equations), so only $n - p$ of them are free. Dividing by $n$ would *under*estimate $\sigma^2$ systematically.

### 6.3 Sampling distribution under normality (adding A5)

Adding A5 ($\varepsilon \sim \mathcal{N}(0, \sigma^2 I_n)$) makes $\hat{\theta}$ a linear function of a Gaussian, hence Gaussian itself:

> **Theorem 6.3.** Under A1 + A2 + A3 + A4 + A5,
>
> $$\hat{\theta} \sim \mathcal{N}(\theta,\; \sigma^2 (X^\top X)^{-1}), \quad \text{and} \quad \text{RSS}/\sigma^2 \sim \chi^2(n-p)$$
>
> with $\hat{\theta}$ and RSS independent.

**Proof sketch.** From (3.1), $\hat{\theta} = \theta + (X^\top X)^{-1} X^\top \varepsilon$ is an affine function of the Gaussian vector $\varepsilon$, hence Gaussian with the mean and covariance computed in §3 and §4. The residual vector $\hat{r} = (I_n - H)\varepsilon$ is also Gaussian; $(I_n - H)$ is an orthogonal projection of rank $n - p$, so $\|\hat{r}\|^2 / \sigma^2 = \varepsilon^\top (I_n - H) \varepsilon / \sigma^2$ has a $\chi^2(n-p)$ distribution. Independence of $\hat{\theta}$ and $\hat{r}$ follows from the orthogonality of their generating projections ($H$ and $I_n - H$), which makes their covariance zero — and for Gaussians, zero covariance implies independence. $\blacksquare$

**This is the key result for the rest of the notebook.** Sections 7 and 8 are just corollaries: confidence intervals follow from "$\hat{\theta}$ is normal"; the t-test follows from "$\hat{\theta}$ normal + $s^2$ independent $\chi^2/(n - p)$".

## 7. Confidence intervals

### 7.1 Definition (standard error)

The **standard error** of $\hat{\theta}_j$ is the square root of its variance estimate:

> $$\text{SE}(\hat{\theta}_j) := s \cdot \sqrt{[(X^\top X)^{-1}]_{jj}}, \quad j = 1, \ldots, p. \tag{7.1}$$

### 7.2 Theorem (t-pivot)

> **Theorem 7.2.** Under A1–A5, for each coordinate $j$,
>
> $$\frac{\hat{\theta}_j - \theta_j}{\text{SE}(\hat{\theta}_j)} \sim t(n-p)$$

**Proof.** Let $V_j := [(X^\top X)^{-1}]_{jj}$. By Theorem 6.3, $(\hat{\theta}_j - \theta_j)/(\sigma \sqrt{V_j}) \sim \mathcal{N}(0, 1)$ and is independent of $(n - p) s^2/\sigma^2 \sim \chi^2(n-p)$. The ratio of a standard normal to $\sqrt{\chi^2/\text{df}}$ is the textbook definition of Student's $t$ with $\text{df} = n - p$:

$$\frac{(\hat{\theta}_j - \theta_j) / (\sigma \sqrt{V_j})}{\sqrt{s^2 / \sigma^2}} = \frac{\mathcal{N}(0,1)}{\sqrt{\chi^2(n-p)/(n-p)}} \sim t(n-p). \quad \blacksquare$$

### 7.3 Construction

Inverting the t-pivot: a $(1 - \alpha)$ confidence interval for $\theta_j$ is

> $$\hat{\theta}_j \pm t_{1-\alpha/2}(n-p) \cdot \text{SE}(\hat{\theta}_j) \tag{7.2}$$

**Interpretation.** Across repeated draws of y (with X fixed), the random interval in (7.2) covers the fixed true $\theta_j$ with probability $1 - \alpha$.

### 7.4 Monte Carlo coverage check

If the CI is built correctly, 95 % of intervals should cover the true value. We measure this empirically.

In [ ]:
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha / 2, df=n - p)

covered = np.zeros(p, dtype=int)
XtX_inv = np.linalg.inv(X.T @ X)
diag_V = np.diag(XtX_inv)

for m in range(M):
    eps = rng.normal(0, sigma, size=n)
    y = X @ theta_true + eps
    theta_hat = XtX_inv @ X.T @ y
    resid = y - X @ theta_hat
    s2 = resid @ resid / (n - p)
    se = np.sqrt(s2 * diag_V)
    lo = theta_hat - t_crit * se
    hi = theta_hat + t_crit * se
    covered += ((lo <= theta_true) & (theta_true <= hi)).astype(int)

print(f"target coverage  = {1 - alpha:.2f}")
print(f"empirical coverage per coefficient:")
for j in range(p):
    print(f"  theta_{j}:  {covered[j] / M:.4f}")

**Reading.** With M = 5000 the empirical coverage of each 95% CI should fall within Monte-Carlo error ($\approx$ 0.6 pp) of 0.95. If we **violated** the assumptions (e.g. fat-tailed noise, heteroscedasticity), this number would drift away from 0.95 — a quick diagnostic in §10.

## 8. Hypothesis tests

### 8.1 The single-coefficient t-test

The most common test in regression: is coefficient $j$ non-zero, i.e. does feature $j$ carry any signal?

> $$H_0 : \theta_j = 0 \quad \text{vs.} \quad H_1 : \theta_j \neq 0.$$

Test statistic:

> $$t_j := \frac{\hat{\theta}_j}{\text{SE}(\hat{\theta}_j)} \tag{8.1}$$

Under $H_0$, $t_j \sim t(n-p)$ by Theorem 7.2. Reject at level $\alpha$ if $|t_j| > t_{1-\alpha/2}(n-p)$.

The **p-value** is

> $$p_j = 2 \cdot \mathbb{P}_{T \sim t(n-p)}(T > |t_j|)$$

### 8.2 The F-test for a sub-model

Sometimes we want to test that *several* coefficients are simultaneously zero, e.g. $H_0 : \theta_2 = \theta_3 = \theta_4 = 0$. Let $\text{RSS}_{\text{full}}$ be the residual sum of squares of the full model and $\text{RSS}_{\text{red}}$ that of the *reduced* model with $q < p$ free parameters. Define

> $$F := \frac{(\text{RSS}_{\text{red}} - \text{RSS}_{\text{full}}) / (p-q)}{\text{RSS}_{\text{full}} / (n-p)} \tag{8.2}$$

Under $H_0$ + A1–A5, $F \sim F(p-q, n-p)$. Reject for large $F$.

**Special case.** When $q = 1$ (only the intercept remains), this is the **overall F-test** of "any feature carries signal" — what most software reports as F-statistic at the bottom of a regression table.

*(Both tests are derived in any standard text, e.g. Hastie–Tibshirani–Friedman* Elements of Statistical Learning *§3.2; we omit the derivation to keep the focus on what each test tests.)*

## 9. The $R^2$ decomposition

Pythagoras (Corollary 5.4 of `02_mathematics.ipynb`) gave

> $$\|y\|^2 = \|\hat{y}^*\|^2 + \|r^*\|^2.$$

When the design includes an intercept column $\mathbb{1}$, an analogous decomposition holds for the **centred** target $\tilde{y} := y - \bar{y} \cdot \mathbb{1}$:

> $$\|\tilde{y}\|^2 = \|\hat{y} - \bar{y} \mathbb{1}\|^2 + \|\hat{r}\|^2 \tag{9.1}$$

Naming each piece:

| Symbol | Name | Formula |
|---|---|---|
| TSS | Total sum of squares | $\sum_i (y_i - \bar{y})^2$ |
| ESS | Explained sum of squares | $\sum_i (\hat{y}_i - \bar{y})^2$ |
| RSS | Residual sum of squares | $\sum_i (y_i - \hat{y}_i)^2$ |

(9.1) says **TSS = ESS + RSS**.

### 9.1 Definition (coefficient of determination)

> $$R^2 := 1 - \frac{\text{RSS}}{\text{TSS}} = \frac{\text{ESS}}{\text{TSS}} \tag{9.2}$$

**Range.** From TSS = ESS + RSS and ESS, RSS $\ge 0$: $0 \le R^2 \le 1$, with $R^2 = 1$ iff RSS = 0 (perfect fit on the training set) and $R^2 = 0$ iff RSS = TSS (the model does no better than predicting the mean).

**Why bother?** $R^2$ is *scale-free*. MSE depends on the unit of y; $R^2$ says what fraction of the variance of y the model accounts for, regardless of the unit.

### 9.2 Caveats

1. **$R^2$ mechanically increases with more features.** Even random columns added to X lower the RSS (the minimum of a quadratic over a *larger* affine space cannot increase). The **adjusted $R^2$** corrects for this:

   > $$R^2_{\text{adj}} = 1 - \frac{\text{RSS} / (n-p)}{\text{TSS} / (n-1)}$$

2. **$R^2$ is a training-set quantity.** It says nothing about generalisation. A model with $R^2 = 0.99$ on training and $R^2 = 0.10$ on a held-out set has overfit — see `00_foundations/04_model_evaluation/`.

In [ ]:
# Single fit on one realisation of y to compute R^2 and TSS = ESS + RSS numerically.
eps = rng.normal(0, sigma, size=n)
y = X @ theta_true + eps
theta_hat = np.linalg.solve(X.T @ X, X.T @ y)
y_hat = X @ theta_hat
y_bar = y.mean()

TSS = float(np.sum((y - y_bar) ** 2))
ESS = float(np.sum((y_hat - y_bar) ** 2))
RSS = float(np.sum((y - y_hat) ** 2))
R2  = 1 - RSS / TSS

print(f"TSS         = {TSS:.4f}")
print(f"ESS         = {ESS:.4f}")
print(f"RSS         = {RSS:.4f}")
print(f"ESS + RSS   = {ESS + RSS:.4f}   (should equal TSS)")
print(f"R^2         = {R2:.4f}")
print(f"identity holds within {abs(ESS + RSS - TSS):.2e}")

## 10. When the assumptions break

Every theorem above carries the tag *under A1–A5*. The table catalogues what goes wrong when each assumption is violated, and points to the diagnostic that detects it.

| Violation | Symptom | Damage | Diagnostic |
|---|---|---|---|
| **A1 fails** ($\operatorname{rank}(X) < p$, multicollinearity) | Two or more columns of X near-linearly dependent | $(X^\top X)^{-1}$ blows up → huge SEs, unstable signs | Variance inflation factors (VIF); condition number of X; correlation matrix |
| **A2 fails** ($\mathbb{E}[\varepsilon \mid X] \neq 0$, e.g. omitted variable, simultaneity) | Bias in $\hat{\theta}$ | $\hat{\theta}$ inconsistent — no amount of data fixes it | Residual plots vs. omitted predictors; instrumental-variable methods |
| **A3 fails** (heteroscedasticity, $\operatorname{Var}(\varepsilon_i)$ depends on $x_i$) | Errors fan out in residual-vs-fitted plot | OLS still unbiased, but $\operatorname{Var}(\hat{\theta}) \neq \sigma^2 (X^\top X)^{-1}$ → CIs and p-values wrong | Breusch–Pagan / White test; heteroscedasticity-robust (HC) standard errors |
| **A4 fails** (autocorrelation, e.g. time-series) | Adjacent residuals correlated | Same as A3: SEs wrong | Durbin–Watson test; Newey–West standard errors |
| **A5 fails** (non-normal noise) | Heavy-tailed or skewed residuals | $\hat{\theta}$ still has correct mean & variance (Theorems 3.1, 4.1 only need A1–A4); but the $t$ / $F$ **exact** distributions in §7–§8 break | QQ-plot of residuals; rely on the **asymptotic** Gaussian via the CLT when n is large enough |

**The mental model.** A1 + A2 deliver **unbiasedness** (Theorem 3.1). A3 + A4 deliver the **variance formula** (Theorem 4.1) and Gauss–Markov optimality (Theorem 5.2). A5 delivers the **exact small-sample distribution** (Theorem 6.3) on which t-tests, F-tests, and CIs depend.

When A5 fails but A1–A4 hold and n is large, the **central limit theorem** rescues CIs and tests *asymptotically* — replace t-quantiles with normal quantiles. When A3 or A4 fail, the point estimates are still good but the *uncertainty quantification* is corrupted; the textbook fix is robust standard errors.

Algorithms in later notebooks of module 01 attack the A1 failure: **Ridge regression** (`03_ridge_regression/`) replaces $(X^\top X)^{-1}$ with $(X^\top X + \lambda I_p)^{-1}$, trading a little bias for a large variance reduction; **Lasso** (`04_lasso_regression/`) does the same and additionally zeroes out coefficients.

## Takeaway

- **Setup.**   X fixed, $y = X\theta + \varepsilon$ with $\varepsilon$ random. $\hat{\theta}$ random because $\hat{\theta} = \theta + (X^\top X)^{-1} X^\top \varepsilon$ (sampling identity (3.1)).
- **Unbiasedness (Theorem 3.1).**   Under A1 + A2: $\mathbb{E}[\hat{\theta}] = \theta$.
- **Variance (Theorem 4.1).**   Under A1 + A2 + A3 + A4: $\operatorname{Var}(\hat{\theta}) = \sigma^2 (X^\top X)^{-1}$.
- **Gauss–Markov (Theorem 5.2).**   OLS is the unique BLUE: minimum variance among all linear unbiased estimators.
- **$\sigma^2$ estimator (Theorem 6.2).**   $s^2 = \text{RSS} / (n - p)$ is unbiased; the $n - p$ divisor counts residual degrees of freedom.
- **Sampling distribution (Theorem 6.3).**   Adding A5: $\hat{\theta} \sim \mathcal{N}(\theta, \sigma^2 (X^\top X)^{-1})$, $\text{RSS}/\sigma^2 \sim \chi^2(n-p)$, and the two are independent.
- **Inference.**   $t_j = \hat{\theta}_j / \text{SE}(\hat{\theta}_j) \sim t(n-p)$ → confidence intervals (7.2) and the coefficient t-test (8.1); F-statistic (8.2) for sub-model tests.
- **$R^2$ (eqs. 9.1–9.2).**   TSS = ESS + RSS (centred Pythagoras); $R^2 = 1 - \text{RSS} / \text{TSS}$ is the variance fraction explained on the training set. Adjusted $R^2$ penalises model size.
- **Failure modes (§10).**   A1 fails → variance inflation; A2 fails → bias; A3 / A4 fail → wrong standard errors; A5 fails → exact tests invalid, asymptotic tests still work.

Next: `05_hands_on_programming.ipynb` — implement everything we have derived (closed-form OLS, gradient descent OLS, residual variance, standard errors, $R^2$) from scratch in NumPy, then cross-check against `sklearn.linear_model.LinearRegression` on a real dataset.